In [5]:
import json
import os
import pandas as pd
import yfinance as yf
from datetime import date
from scipy.stats import gmean
from yfetch import delay, get_stock_history, get_stock_name, get_stock_metadata

risk_free_return = 4/100
history_weeks = 3 * 52 # 3Y
change_weeks = 52 # 1Y
range_window = 4 # ~ month

symbols = [
    'MAGS',
    'FNGS',
    'IGM',
    'SPYG',
    'SPMO',
    'IWMO.MI',
    'TSLA',
    'NVDA',
    'AVGO',
    'SMH',
    'USD',
    'TDIV.AS',
    'ESIF.DE',
    'DFEN.DE',
    'XDWI.DE',
    '4GLD.DE',
    'BRK-B',
    'NFLX',
    'PLTR',
    'NET',
    'ISRG',
    'SHOP',
    'DAPP',
    'BITQ',
    'UFO',
    'ROKT',
    'JEDI.DE',
    'SPCX',
    'RKLB',
    'PL',
    'NASA',
    'NUKZ',
    'NLR',
    'AIPO',
    'QTUM',
    'AGIX',
    'CHAT',
    'BAI',
    'ARKG',
    'PPA',
    'QQQ',
    'COIN',
    'NKE',
    'STLA',
    'AMAT',
]

revenue_cache_file = 'data/revenue_cache.json'
revenue_cache_days = 7 # revenue is reported quarterly, no need to check often

try:
    with open(revenue_cache_file) as f:
        revenue_cache = json.load(f)
except FileNotFoundError:
    revenue_cache = {}

def fetch_revenue_growth(symbol):
    """(year over year growth of the latest quarterly revenue, quarter end date),
    (None, None) if unavailable."""
    delay()
    qis = yf.Ticker(symbol).quarterly_income_stmt
    if 'Total Revenue' not in qis.index:
        print(f'No revenue for {symbol}')
        return None, None
    rev = qis.loc['Total Revenue'].dropna().sort_index(ascending=False)
    if len(rev) < 5:
        print(f'Not enough revenue history for {symbol}: {len(rev)} quarters, need 5')
        return None, None
    return round(float(rev.iloc[0] / rev.iloc[4] - 1), 4), str(rev.index[0].date())

def format_revenue(growth, quarter):
    return 'n/a' if growth is None else f'{growth:.2%} ({quarter})'

def get_revenue_growth(symbol, cache_days=revenue_cache_days):
    """Revenue growth from data/revenue_cache.json, refetched once it goes stale."""
    if get_stock_metadata(symbol).get('instrumentType') != 'EQUITY':
        return None # ETFs and the like have no revenue

    today = date.today()
    cached = revenue_cache.get(symbol)
    if cached and (today - date.fromisoformat(cached['fetched'])).days < cache_days:
        return cached['growth']

    growth, quarter = fetch_revenue_growth(symbol)
    if cached and (growth, quarter) != (cached['growth'], cached['quarter']):
        print(f'{symbol} revenue: {format_revenue(cached["growth"], cached["quarter"])}'
              f' => {format_revenue(growth, quarter)}')

    revenue_cache[symbol] = {'growth': growth, 'quarter': quarter, 'fetched': today.isoformat()}
    os.makedirs(os.path.dirname(revenue_cache_file), exist_ok=True)
    with open(revenue_cache_file, 'w') as f:
        json.dump(revenue_cache, f, indent=2, sort_keys=True)
    return growth

def temperature(series):
    """Share of past values at or below the last one, None if the series is empty."""
    series = series.dropna()
    if len(series) == 0:
        return None
    return (series <= series.iloc[-1]).mean()

def sma_temperature(symbol, window=200):
    """Temperature of the distance to the SMA over daily history."""
    daily = get_stock_history(symbol, period='5y', interval='1d', cache_days=5)
    sma = daily.Close.rolling(window=window).mean()
    if sma.dropna().empty:
        print(f'Not enough history for {symbol}: {len(daily)} days, need {window}')
        return None
    return temperature(daily.Close / sma - 1)

rows = []
for symbol in symbols:
    history = get_stock_history(symbol, interval='1wk')
    history = history.tail(history_weeks)

    if len(history) < history_weeks:
        print(f'Not enough history for {symbol}: {len(history)} weeks, need {history_weeks}')
        gmean_change = std = sharpe = None
    else:
        changes = history.Close.pct_change(periods=change_weeks, fill_method=None).dropna()
        gmean_change = gmean(1 + changes) - 1 # geometric mean of changes
        std = changes.std()
        sharpe = (gmean_change - risk_free_return) / std

    high = history.High.rolling(range_window).max()
    low = history.Low.rolling(range_window).min()
    range = ((high - low) / (high + low) * 2).dropna() # relative range per window

    rows.append({
        'symbol': symbol,
        'name': get_stock_name(symbol),
        'price': history.Close.iloc[-1], # last close
        'weeks': len(history),
        'gmean': gmean_change,
        'std': std,
        'sharpe': sharpe,
        'range': range.mean(),
        'revenue': get_revenue_growth(symbol),
        'T200': sma_temperature(symbol, window=200),
    })

df = pd.DataFrame(rows)
f = f'data/folio.csv'
df.to_csv(f, index=False)
print(f'Saved to {f} ({len(df)} rows)')

df = df.reset_index(drop=True)
for col in ['gmean', 'std', 'range', 'revenue', 'T200']:
    df[col] = df[col].map(lambda v: None if pd.isna(v) else f'{v:.2%}')
df

Fetched history for MAGS (835 rows)
Fetched history for FNGS (1255 rows)
Fetched history for IGM (1255 rows)
Fetched history for SPYG (1255 rows)
Fetched history for SPMO (1255 rows)
Fetched history for IWMO.MI (1269 rows)
Fetched history for TSLA (1255 rows)
Fetched history for NVDA (1255 rows)
Fetched history for AVGO (1255 rows)
Fetched history for SMH (1255 rows)
Fetched history for USD (1255 rows)
Fetched history for TDIV.AS (1280 rows)
Fetched history for ESIF.DE (1275 rows)
Fetched history for DFEN.DE (849 rows)
Fetched history for XDWI.DE (1273 rows)
No revenue for 4GLD.DE
Fetched history for 4GLD.DE (1273 rows)
Fetched history for BRK-B (1255 rows)
Fetched history for NFLX (1255 rows)
Fetched history for PLTR (1255 rows)
Fetched history for NET (1255 rows)
Fetched history for ISRG (1255 rows)
Fetched history for SHOP (1255 rows)
Fetched history for DAPP (1255 rows)
Fetched history for BITQ (1255 rows)
Fetched history for UFO (1255 rows)
Fetched history for ROKT (1255 rows)
Fet

,symbol,name,price,weeks,gmean,std,sharpe,range,revenue,T200
0,MAGS,Roundhill Magnificent Seven ETF,69.139999,156,34.44%,16.78%,1.813527,10.54%,None,25.47%
1,FNGS,MicroSectors FANG+ ETN,80.779999,156,31.56%,14.99%,1.838612,10.65%,None,64.11%
2,IGM,iShares Expanded Tech Sector ETF,163.720001,156,31.42%,14.56%,1.883844,9.88%,None,78.03%
3,SPYG,State Street SPDR Portfolio S&P 500 Growth ETF,123.019997,156,26.24%,9.66%,2.302270,7.73%,None,74.34%
4,SPMO,Invesco S&P 500 Momentum ETF,149.690002,156,34.54%,13.60%,2.245589,8.60%,None,77.18%
5,IWMO.MI,iShares Edge MSCI World Momentum Factor UCITS ETF,100.160004,156,19.70%,14.08%,1.114965,7.56%,None,79.91%
6,TSLA,"Tesla, Inc.",328.579987,156,38.93%,31.38%,1.113322,23.03%,25.52%,17.42%
7,NVDA,NVIDIA Corporation,223.960007,156,65.31%,62.92%,0.974490,19.12%,85.23%,39.77%
8,AVGO,Broadcom Inc.,427.760010,156,76.68%,29.37%,2.474357,19.66%,47.87%,39.58%
9,SMH,VanEck Semiconductor ETF,582.700012,156,47.39%,42.34%,1.024727,14.80%,None,75.19%
